# SAM + BAMS Deposition Log

Use this notebook when you apply:
- **SAM** (PFBT self-assembled monolayer): UVO + immersion
- **BAMS** (bar-assisted meniscus shearing): semiconductor deposition

Select the samples you're processing below.

In [1]:
import sys, os, json
from datetime import datetime
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '../..'))
sys.path.insert(0, PROJECT_ROOT)

from src.database import init_db, list_samples, add_step, get_sample, update_step
import pandas as pd

init_db()

samples_all = list_samples()
df = pd.DataFrame(samples_all)
df[['label', 'step_status']]

,label,step_status
0,260702_W1_P1,"photolithography:completed, developing:complet..."
1,260702_W1_P2,"photolithography:completed, developing:complet..."
2,260702_W1_P3,"photolithography:completed, developing:complet..."
3,260702_W1_P4,"photolithography:completed, developing:complet..."
4,260626_W1_P1,"photolithography:completed, evaporation:comple..."
5,260626_W1_P2,"photolithography:completed, evaporation:comple..."
6,260626_W1_P3,"photolithography:completed, evaporation:comple..."
7,260626_W1_P4,"photolithography:completed, evaporation:comple..."


## 1. Select Samples to Process

Enter the sample labels you're working on in this session.

In [2]:
SAMPLES_TO_PROCESS = [
    '260626_W1_P1',
    # '260626_W1_P2',  # Uncomment as needed
]

sample_ids = []
for label in SAMPLES_TO_PROCESS:
    matches = [s for s in samples_all if s['label'] == label]
    if matches:
        sample_ids.append(matches[0]['id'])
        print(f'Found: {label}')
    else:
        print(f'NOT FOUND: {label}')

print(f'\nProcessing {len(sample_ids)} sample(s).')

Found: 260626_W1_P1

Processing 1 sample(s).


## 2. UVO + SAM

UV Ozone treatment → SAM immersion → Rinse

In [3]:
SAM_PARAMS = {
    'sam_type': 'PFBT',
    'concentration_ul_ml': 2,
    'ipa_volume_ml': 20,
    'pfbt_volume_ul': 40,
    'uvo_time_min': 25,
    'immersion_time_min': 15,
}
SAM_NOTES = ''

for sid in sample_ids:
    add_step(sid, 'sam', status='completed',
             params=SAM_PARAMS, notes=SAM_NOTES)

print(f'SAM registered for {len(sample_ids)} sample(s).')
print('Store samples in IPA until BAMS. Do NOT leave long in IPA.')

SAM registered for 1 sample(s).
Store samples in IPA until BAMS. Do NOT leave long in IPA.


## 3. BAMS Deposition

Bar-assisted meniscus shearing of semiconductor ink.

**Pre-BAMS checklist:**
- [ ] Hotplate at 105°C
- [ ] Bar preheated 30-60 min
- [ ] Semiconductor vial on hotplate
- [ ] CB between hotplate and sample for fixation
- [ ] 30-40 µL per device

In [4]:
BAMS_PARAMS = {
    'bar_number': 3,
    'temp_C': 105,
    'speed_mm_s': 10,
    'motor_steps_s': 2778,
    'volume_ul_per_device': 35,
    'ink_batch': '',  # Fill in the ink batch name
}
BAMS_NOTES = ''

for sid in sample_ids:
    add_step(sid, 'bams', status='completed',
             params=BAMS_PARAMS, notes=BAMS_NOTES)

print(f'BAMS registered for {len(sample_ids)} sample(s).')
print('Store devices in vacuum/glovebox after deposition.')

BAMS registered for 1 sample(s).
Store devices in vacuum/glovebox after deposition.


## 4. Functionalization (Optional)

If you're doing surface functionalization of specific samples.

In [5]:
FUNC_SAMPLES = [
    # '260626_W1_P4',  # Pieces being functionalized
]

FUNC_PARAMS = {
    'type': '',  # e.g., 'protein_A_immobilization', 'antibody_binding'
    'concentration': '',
    'incubation_time_min': 0,
    'buffer': '',
}
FUNC_NOTES = ''

for label in FUNC_SAMPLES:
    matches = [s for s in samples_all if s['label'] == label]
    if matches:
        add_step(matches[0]['id'], 'functionalization', status='completed',
                 params=FUNC_PARAMS, notes=FUNC_NOTES)
        # Update sample functionalization field
        print(f'Functionalized: {label}')
    else:
        print(f'NOT FOUND: {label}')

print('\nDone! Samples ready for measurement.')


Done! Samples ready for measurement.
